# 两个完整流程的对比：本仓库 `reconstruct_from_slp` vs medaka-3d 生产链（20260112_P_1_1_FCMC）

之前的 `20260112_P_1_1_FCMC_medaka3d_refractive_3d.ipynb` 只把 **逐帧跨视角配对** 换成了 medaka-3d，连轨迹和身份仍用本仓库的。
这本 notebook 跑 **medaka-3d 自己的整条生产链**（它的 `docs/TRACKING_PIPELINE.md`），一个文件都不改，
再把最终结果做成和本仓库同格式的审阅包，以人工审阅结果为标准答案比较两个完整流程。

| | 本仓库（old） | medaka-3d 完整流程（m3dfull） |
|---|---|---|
| 检测 | s14 bottom-up，`--max_instances 6` | **同一份 `.slp`**（medaka-3d 原本用同一模型、不限实例数） |
| 2D | 只给顶视角做 `track_simple_nn`（ID 绑定用） | **c27**：三个相机各自 2D 追踪（sleap_nn Tracker，不限轨迹数，串行） |
| 配对 + 时间 | 每帧选三元组 → 3D 匈牙利连轨迹（8 / 80 mm，100 帧） | **c28**：整段录像上的 Viterbi，配对和时间连续性一起解（7 个 λ） |
| 修补 | — | **c33** 剔除坏射线 → **c30 v4** 槽拼接（证据阶梯）→ **c34** 往返翻转修复 |
| 最终身份 | `bind_topcam_id` → `final_fish_id` | 3 个永久槽 ID（`C34-label`），**没有 F/M1/M2 识别器、没有 bind** |
| 标定 | `calibration_results/fixed_refractive_192x192x186` | **逐字节相同**的同一份（步骤 2 用 sha256 校验） |

| 步骤 | 做什么 | 用时 |
|---|---|---|
| 1–2 | 参数；定位两个仓库、路径、标定校验 | 秒级 |
| 3 | 入口转换：本仓库 `.slp` → medaka-3d 的 c01 检测缓存 | ~1.5 分钟（已建好则跳过） |
| 4 | 短窗口验证 c27（2 万帧）的结果 | 读结果 |
| 5 | 后台启动整条链 c27 → c28 → c33 → c30 → c34 | ~1.5 h + ~2 h + 几分钟 |
| 6 | 进度 | 随时重跑 |
| 7 | 出口转换 + 生成审阅包 `<ROUND_TAG>_m3dfull` | ~10 分钟 |
| 8 | 对比数据：reviewed / old / m3dfull | ~2 分钟 |
| 9 | 身份：切换次数（主指标）、分歧帧 | 秒级 |
| 10 | λ 扫描：7 个 λ 的切换次数 | ~1 分钟 |
| 11 | 几何（不依赖参考） | 秒级 |
| 12 | 打开对比 GUI | — |

**背后的脚本**（逻辑都在 `scripts/`，notebook 只调用）：
- `scripts/reconstruction_3d/medaka3d_chain.py` — 入口转换（c01 缓存）、标定路径重定位、按顺序运行链（已完成的步骤自动跳过）、出口转换、λ 扫描
- `scripts/review/review_data.py` — 审阅包；`scripts/review/recon_compare.py` — 对比数据、`switch_table` / `geometry_table` / `error_table`
- medaka-3d 的 `tools/c27…c34`：**原样运行**，只有 c28 / c33 经 `medaka3d_chain run` 启动，把写死的标定路径换成本机路径

**读结果前要知道的**：
- **检测不完全对等**：本仓库推理限了 6 个实例，medaka-3d 原本不限（每帧最多见过 8 个）。真要完全对等，得不限实例重跑推理
- **c30 v4 的阈值是在 K_1_2 上标定的**（2.5 mm / 30 mm）。用在 P_1_1 上是迁移测试
- **λ 不选**（medaka-3d 的规则）：审阅包用 0.767（它渲染用的那个），λ 扫描给出全部 7 个；结论只取所有 λ 都成立的部分
- **身份的主指标是切换次数**，不是“名字错的帧数”：后者主要反映第一次永久换错发生得多早
- **几何偏向 old**：审阅是在 old 上改的，没改的地方 reviewed 就等于 old。几何只看不依赖参考的指标

## 步骤 1 — 参数

只需要改这一格。
- `RUN_CHAIN`：为 `True` 时步骤 5 才真正在后台启动整条链（防止“全部运行”误启动）；已完成的步骤会自动跳过，中断后再启动即可接着跑
- `RUN_EXPORT`：链完成后导出并生成审阅包；已存在则跳过，除非 `OVERWRITE`

In [1]:
EXPERIMENT = "20260112_P_1_1_FCMC"
ROUND_TAG  = "s14_bottomup_3node_reviewed"     # 本仓库 SLEAP 输出的轮次（两个流程读同一份检测）
OUT_TAG    = f"{ROUND_TAG}_m3dfull"            # medaka-3d 完整流程的结果 / 审阅包名
LAMBDA     = 0.767                              # 审阅包用的 λ（medaka-3d 渲染用的那个，不是“选出来的”）
BRIDGE_TAG = f"{ROUND_TAG}_medaka3d"            # 只换逐帧配对的桥接结果（拆解差异用；没有就跳过）

MEDAKA3D_REPO = None    # None = 本仓库旁边的 ../medaka-3d
PATHS_CONFIG  = None    # None = 自动选择
RUN_CHAIN  = False      # 步骤 5：后台启动整条链
RUN_EXPORT = True       # 步骤 7：链完成后导出 + 生成审阅包
OVERWRITE  = False
OPEN_GUI   = False      # 步骤 12

## 步骤 2 — 定位两个仓库、路径配置，校验标定

- 输入：当前目录、`configs/paths.<用户名>.toml`、`../medaka-3d`（及其 `configs/paths.local.toml`）
- 输出：`REPO`、`PATHS`、`M3D`、`M3P`（medaka-3d 的路径）、`SLP_DIR`、`OUT_DIR`、`REVIEW_ROOT`、`LOG_DIR`
- medaka-3d 的工具把标定写死为 `Z:\StephensU\medaka\calibration\…`（写它的那台电脑的盘符）。`local_calibration` 换到本机挂载点，
  并逐个比较 `cam*_bundle_refractive.json` 的 sha256，和本仓库的标定不一致就报错

In [2]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "scripts" / "common" / "local_paths.py").exists():
            return d
    raise FileNotFoundError(f"找不到仓库根目录（从 {start} 向上没有 pyproject.toml）")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.common.local_paths import load_local_paths
from scripts.sleap.batch_bottomup_track import default_paths_config
from scripts.reconstruction_3d import medaka3d_chain as M3C

CFG = Path(PATHS_CONFIG) if PATHS_CONFIG else default_paths_config()
PATHS = load_local_paths(CFG)
if PATHS is None:
    raise FileNotFoundError(f"路径配置不存在：{CFG}（从 configs/paths.example.toml 复制一份再改）")
M3D = M3C.medaka3d_repo(MEDAKA3D_REPO)
M3P = M3C.m3d_paths(M3D)
CAL = M3C.local_calibration(M3D)

SLP_DIR = PATHS.sleap_output_dir / EXPERIMENT / ROUND_TAG
OUT_DIR = PATHS.reconstruction_output_dir / EXPERIMENT / OUT_TAG
REVIEW_ROOT = PATHS.data_root / "review_packages"
LOG_DIR = M3P.work_dir / "medaka3d_chain_logs" / EXPERIMENT
print("仓库根目录   :", REPO, " 版本", M3C._git_rev(REPO))
print("medaka-3d    :", M3D, " 版本", M3C._git_rev(M3D))
print("medaka-3d work:", M3P.work_dir)
print("标定（medaka-3d 用）:", CAL, " —— 与本仓库", M3C.CALIBRATION_DIR.name, "逐字节一致 ✅")
print("检测 .slp    :", SLP_DIR)
print("Python       :", sys.executable)
if not Path(sys.executable).resolve().is_relative_to((REPO / ".venv").resolve()):
    print("⚠️ 当前内核不是本仓库的 .venv —— 请在 Jupyter 里切换内核")

仓库根目录   : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline  版本 d208c31
medaka-3d    : C:\Users\ReiterU\Documents\GitHub\medaka-3d  版本 50d56d0
medaka-3d work: Z:\medaka\work
标定（medaka-3d 用）: Z:\medaka\calibration\fixed_refractive_192x192x186\phase1-calibration-preserve-20260802T180200JST-60e13a7746d1\calibration  —— 与本仓库 fixed_refractive_192x192x186 逐字节一致 ✅
检测 .slp    : \\bucket.oist.jp\bucket\StephensU\medaka\sleap_output\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed
Python       : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\.venv\Scripts\python.exe


## 步骤 3 — 入口转换：本仓库 `.slp` → medaka-3d 的 c01 检测缓存

- 输入：`SLP_DIR/{top_cam,side1_cam,side2_cam}.slp`
- 输出：`<work>/c01_s16_predictions/bottomup/<EXPERIMENT>__<view>.npz` + `c01_bottomup__<EXPERIMENT>.provenance.json`（记录推理命令）
- 用 c01 自己的 `read_pair` / `save`：每帧都核对视频路径里的实验名和视角，`.slp` 带 tracks 也会拒绝
- 缓存比 `.slp` 新就跳过

In [3]:
import json

import pandas as pd

M3C.build_detection_cache(EXPERIMENT, SLP_DIR, m3d=M3D)
prov = json.loads((M3P.work_dir / "c01_s16_predictions" / M3C.RUN / f"c01_{M3C.RUN}__{EXPERIMENT}.provenance.json")
                  .read_text(encoding="utf-8"))
print("状态:", prov["status"], " 建于", prov["finished_at"])
display(pd.DataFrame(prov["params"]["summary"]).T.assign(
    **{"=3 %": lambda d: (100 * d["eq3"] / prov["params"]["n_frames"]).round(1)}))
print("推理命令（top_cam）:", prov["params"]["inference_command__top_cam"][-60:])

  all 3 views cached and newer than their .slp; kept (provenance untouched)
状态: ok  建于 2026-10-28T06:32:50.298038+00:00


,instances,frames_empty,eq3,max_per_frame,=3 %
top_cam,1042426,178,326803,6,93.7
side1_cam,1025627,0,319693,6,91.7
side2_cam,1030102,0,324475,6,93.1


推理命令（top_cam）: m.partial.slp --device cuda --batch_size 4 --max_instances 6


## 步骤 4 — 短窗口验证 c27（第 171800–191800 帧）

- 已经在命令行跑过一次，确认入口转换的缓存能被 medaka-3d 读、2D 追踪能跑通；这里只读结果
- 输出在单独的 `…_serial_frames171800-191800/` 下，不会和全长结果混（c28 只读全长的 `…_serial/`）
- 重跑：`M3C.tool_command("c27", ["--experiment", EXPERIMENT, "--frames", "171800", "191800"])` 给出的命令

In [4]:
win = M3P.work_dir / "c27_uncapped_tracklets_any" / EXPERIMENT / "mouth-pectoral_oks_serial_frames171800-191800"
if (win / "c27.provenance.json").exists():
    p = json.loads((win / "c27.provenance.json").read_text(encoding="utf-8"))
    s = json.loads((win / "c27_summary.json").read_text(encoding="utf-8"))
    print("状态:", p["status"], " 帧", p["params"]["frames"], " 用时", round(p["params"]["wall_seconds"]), "s",
          " 未标记实例（三个相机）:", s["unlabelled_total"])
    display(pd.DataFrame({"未标记实例": s["unlabelled_instances"], "每帧平均状态数": s["mean_states_per_frame"]}))
else:
    print("还没跑短窗口验证；命令：", " ".join(M3C.tool_command("c27", ["--experiment", EXPERIMENT, "--frames", "171800", "191800"])[0]))

状态: ok  帧 [171800, 191800]  用时 79 s  未标记实例（三个相机）: 0


,未标记实例,每帧平均状态数
top_cam,0,1.299203
side1_cam,0,1.290881
side2_cam,0,1.297800


## 步骤 5 — 后台启动整条链（c27 → c28 → c33 → c30 → c34）

- 独立进程，关掉 notebook / 会话也继续跑；pid 记在 `LOG_DIR/chain.pid.json`，日志 `LOG_DIR/<步骤>.log` 和 `chain.log`
- 每一步的完成以它自己的 `<tool>.provenance.json` 里 `status == "ok"` 为准，**已完成的步骤自动跳过**：睡眠 / 重启中断后，再运行本格即可接着跑
- 用时（K_1_2 实测）：c27 ~1.5 h（三个相机三个 CPU 进程，不用 GPU），c28 每个 λ ~18 分钟 × 7，c33 / c30 / c34 各约 1 分钟

In [5]:
import subprocess
import time

import psutil

LOG_DIR.mkdir(parents=True, exist_ok=True)
PID_FILE = LOG_DIR / "chain.pid.json"


def chain_alive():
    if not PID_FILE.exists():
        return None
    info = json.loads(PID_FILE.read_text())
    try:
        proc = psutil.Process(info["pid"])
        return proc if abs(proc.create_time() - info["started"]) < 60 else None
    except psutil.Error:
        return None


if proc := chain_alive():
    print(f"整条链已在运行，PID {proc.pid}，开始于 {time.ctime(proc.create_time())}")
elif not RUN_CHAIN:
    print("未启动：把步骤 1 的 RUN_CHAIN 改成 True 再运行本格")
else:
    cmd = [sys.executable, "-m", "scripts.reconstruction_3d.medaka3d_chain", "chain", EXPERIMENT, "--log-dir", str(LOG_DIR)]
    log = open(LOG_DIR / "chain.log", "a", encoding="utf-8")
    log.write(f"\n# started {time.ctime()}\n# cmd: {' '.join(cmd)}\n")
    log.flush()
    flags = subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS if os.name == "nt" else 0
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, cwd=REPO, creationflags=flags,
                         env=dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1"))
    PID_FILE.write_text(json.dumps({"pid": p.pid, "started": psutil.Process(p.pid).create_time()}))
    print(f"已启动，PID {p.pid}，日志 {LOG_DIR / 'chain.log'}")

未启动：把步骤 1 的 RUN_CHAIN 改成 True 再运行本格


## 步骤 6 — 进度

- 每一步：`ok` / `failed` / `running` / `missing`（来自各自的 provenance）和输出目录
- 下面是正在跑的那一步日志的最后几行（c27 运行中只在结束时打印；c28 每个 λ 结束打印一次）

In [6]:
STEPS = M3C.chain_steps(EXPERIMENT, M3D)
rows = []
for tool, _, out in STEPS:
    status = M3C.step_status(tool, out)
    when = json.loads((out / f"{tool}.provenance.json").read_text(encoding="utf-8")).get("finished_at", "")[:19] \
        if status not in ("missing", "unreadable") else ""
    rows.append({"步骤": tool, "状态": status, "结束 (UTC)": when, "输出": str(out)})
display(pd.DataFrame(rows))
proc = chain_alive()
print("整条链进程:", f"运行中 PID {proc.pid}" if proc else "没有在运行")
current = next((r["步骤"] for r in rows if r["状态"] != "ok"), None)
if current and (LOG_DIR / f"{current}.log").exists():
    print(f"\n--- {current}.log 最后几行 ---")
    print("".join((LOG_DIR / f"{current}.log").read_text(encoding="utf-8", errors="replace").splitlines(True)[-12:]))
CHAIN_DONE = all(r["状态"] == "ok" for r in rows)
print("整条链完成:", CHAIN_DONE)

,步骤,状态,结束 (UTC),输出
0,c27,ok,2026-10-28T07:03:06,Z:\medaka\work\c27_uncapped_tracklets_any\2026...
1,c28,ok,2026-10-28T08:16:34,Z:\medaka\work\c28_lambda_grid_transfer\202601...
2,c33,ok,2026-10-28T08:18:39,Z:\medaka\work\c33_ray_rejection\20260112_P_1_...
3,c30,ok,2026-10-28T08:21:00,Z:\medaka\work\c30_slot_stitch\20260112_P_1_1_...
4,c34,ok,2026-10-28T08:25:48,Z:\medaka\work\c34_microflip\20260112_P_1_1_FC...


整条链进程: 没有在运行
整条链完成: True


## 步骤 7 — 出口转换 + 生成审阅包 `<ROUND_TAG>_m3dfull`

- 输入：`C34-label__<EXPERIMENT>__lam<LAMBDA>.npz`（3 个永久槽 ID、胸鳍锚点 xyz、每个槽在各视角用的检测行号）
- `export_review_inputs` → `OUT_DIR/<OUT_TAG>_best.json`、`_3d.csv`、`_reconstruction_provenance.json`
  - 槽 k = `ID_(k+1)`，全程不换；**不做 bind**，所以没有 `_3d_identified.csv`
  - X/Y/Z = 链的胸鳍锚点，**不插值**（medaka-3d 宁可留空）
  - 检测行号 → 本仓库 `load_2d_slp` 的槽顺序（审阅包按这个顺序取 2D 点）；本仓库判为假阳性而丢掉的检测无法对应，数量记在 provenance
- `build_review_data` → `REVIEW_ROOT/<EXPERIMENT>/<OUT_TAG>/`：和 old 的审阅包**同一套代码**（同样从 2D 重新三角化三个节点、同样的头尾判定和补全），
  所以两边节点几何的差别只来自“哪个检测分给哪条鱼”

In [7]:
from scripts.review.build_packages import source_fps
from scripts.review.review_data import build_review_data, calibration_folder

REVIEW_A = REVIEW_ROOT / EXPERIMENT / ROUND_TAG          # old（已审阅完）
REVIEW_FULL = REVIEW_ROOT / EXPERIMENT / OUT_TAG
REVIEW_BRIDGE = REVIEW_ROOT / EXPERIMENT / BRIDGE_TAG
if not CHAIN_DONE:
    print("整条链还没完成（见步骤 6），先不导出")
elif not RUN_EXPORT:
    print("未导出：RUN_EXPORT = False")
else:
    if OVERWRITE or not (OUT_DIR / f"{OUT_TAG}_best.json").exists():
        t0 = time.time()
        EXP_OUT = M3C.export_review_inputs(EXPERIMENT, SLP_DIR, OUT_DIR, OUT_TAG, lam=LAMBDA, m3d=M3D)
        print(f"导出用时 {time.time() - t0:.0f} s")
    prov = json.loads((OUT_DIR / f"{OUT_TAG}_reconstruction_provenance.json").read_text(encoding="utf-8"))
    print(json.dumps({k: prov[k] for k in ("arm", "medaka3d_rev", "frames_resolved", "fish_frames_present",
                                           "detections_without_slot")}, ensure_ascii=False, indent=1))
    if OVERWRITE or not (REVIEW_FULL / "review_data.npz").exists():
        t0 = time.time()
        build_review_data(EXPERIMENT, OUT_TAG, SLP_DIR, OUT_DIR, REVIEW_FULL,
                          calibration_folder(OUT_DIR, OUT_TAG, PATHS.calibration_model_dir),
                          fps=source_fps(PATHS, EXPERIMENT))
        print(f"审阅包用时 {time.time() - t0:.0f} s → {REVIEW_FULL}")
    else:
        print("审阅包已存在：", REVIEW_FULL)

[export] top_cam: 1,030,132 detections used, 1,351 without a load_2d_slp slot (49 s)


[export] side1_cam: 1,019,212 detections used, 14,967 without a load_2d_slp slot (46 s)


[export] side2_cam: 1,023,641 detections used, 10,796 without a load_2d_slp slot (47 s)


[export] wrote s14_bottomup_3node_reviewed_m3dfull_best.json, s14_bottomup_3node_reviewed_m3dfull_3d.csv -> \\bucket.oist.jp\bucket\StephensU\medaka\reconstruction_3d\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed_m3dfull
导出用时 173 s
{
 "arm": "Z:\\medaka\\work\\c34_microflip\\20260112_P_1_1_FCMC__chain-c28-c33-c30v4-c34\\C34-label__20260112_P_1_1_FCMC__lam0.767.npz",
 "medaka3d_rev": "50d56d0",
 "frames_resolved": 347927,
 "fish_frames_present": 1043781,
 "detections_without_slot": {
  "top_cam": 1351,
  "side1_cam": 14967,
  "side2_cam": 10796
 }
}


[review_data] 20260112_P_1_1_FCMC s14_bottomup_3node_reviewed_m3dfull: frames 0-348599, fish ['ID_1', 'ID_2', 'ID_3']


[review_data] loaded top_cam.slp (348422 frames, 50 s)


... (5 outputs left out)

[reconstruct] completion + display (16 s)
[review_data] reconstruction: 26 head/tail jumps left, nodes from one view 0.388%, body model 0.028% (110 s)


[review_data] node triangulation + projection (116 s)


[review_data] 2 risk windows; done in 352 s


审阅包用时 352 s → \\bucket.oist.jp\bucket\StephensU\medaka\review_packages\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed_m3dfull


## 步骤 8 — 对比数据：reviewed / old / m3dfull

- `reviewed` = 你审阅完的 `final`（身份全程人工确认，作标准答案）；`old` / `m3dfull` = 各自审阅包的 `initial`（审阅前的机器结果）
- 两个机器版本都在分析起点附近、三条鱼分得开且无歧义的那一帧继承名字，之后各走各的轨迹 —— 所以 old 有 bind、m3dfull 没有识别器，这一点不影响比较
- 输出 `REVIEW_A/recon_reviewed_old_m3dfull/`（GUI 的“3D 重建对比”模式读它）；已存在则跳过，除非 `OVERWRITE`
- 另外读入已有的 `recon_reviewed_old_medaka3d/`（桥接版本），用来拆解差异：逐帧配对的影响 vs 整条链的影响

In [8]:
from scripts.review.recon_compare import (RECON_FILE, ReconState, build, default_out_dir, error_table, geometry_table,
                                          summary, switch_table)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
STATE = STATE_BRIDGE = None
if not (REVIEW_FULL / "review_data.npz").exists():
    print("还没有 m3dfull 的审阅包（步骤 7）")
else:
    folders, kinds, names = [REVIEW_A, REVIEW_A, REVIEW_FULL], ["final", "initial", "initial"], ["reviewed", "old", "m3dfull"]
    RECON_DIR = default_out_dir(folders, names)
    if OVERWRITE or not (RECON_DIR / RECON_FILE).exists():
        t0 = time.time()
        build(folders, kinds, RECON_DIR, names=names, log=print)
        print(f"生成 {RECON_DIR / RECON_FILE}（{time.time() - t0:.0f} s）")
    STATE = ReconState(REVIEW_A, RECON_DIR)
    print("参考的名字:", STATE.labels, " 继承名字的帧:", STATE.link_frame, " 分析起点:", STATE.analysis_start)
bridge_dir = default_out_dir([REVIEW_A, REVIEW_A, REVIEW_BRIDGE], ["reviewed", "old", "medaka3d"])
if (bridge_dir / RECON_FILE).exists():
    STATE_BRIDGE = ReconState(REVIEW_A, bridge_dir)
    print("桥接版本对比:", bridge_dir.name)

[reconstruct] head/tail orientation (65 s)


[reconstruct] triangulation: 0.000% node-frames re-triangulated from two views, 0.000% spikes replaced (18 s)


[reconstruct] completion + display (16 s)


[versions] initial tracks (104 s)


[recon_compare] recon_versions.npz: ['reviewed', 'old', 'm3dfull'] (111 s) -> \\bucket.oist.jp\bucket\StephensU\medaka\review_packages\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed\recon_reviewed_old_m3dfull
生成 \\bucket.oist.jp\bucket\StephensU\medaka\review_packages\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed\recon_reviewed_old_m3dfull\recon_versions.npz（111 s）


参考的名字: ['F', 'M1', 'M2']  继承名字的帧: 4152  分析起点: 4152


桥接版本对比: recon_reviewed_old_medaka3d


## 步骤 9 — 身份（以审阅结果为标准答案）

**表 1：切换次数（主指标）**。分析起点之后，每次某个版本和审阅结果之间的对应关系发生变化就算一次（换错，或者换回来）：
- 少于 8 帧的抖动并入前一段；位置太接近、无法判定的帧（margin < 2 mm）沿用上一帧的对应
- `into wrong` / `back to right`：换完之后是错的 / 回到了起点的命名
- `only this (±1 s)`：1 秒内其他版本都没有换 —— 这个方法自己的错，而不是所有方法都难的相遇
- `with F` 等：涉及这条鱼的切换次数
- `medaka3d` 一行来自桥接版本的对比（只换逐帧配对），用来拆解：old → medaka3d 的差 = 逐帧配对的影响，medaka3d → m3dfull 的差 = 2D 先追踪 + Viterbi + 修补的影响

**表 2：名字错的鱼-帧**（`error_table` 的 swap 一行）只作参考：它主要反映第一次永久换错发生得多早。

In [9]:
if STATE is None:
    print("还没有对比数据（步骤 8）")
else:
    tables = [switch_table(STATE)]
    if STATE_BRIDGE is not None:
        tables.insert(0, switch_table(STATE_BRIDGE).query("version == 'medaka3d'"))
    T_SWITCH = pd.concat(tables, ignore_index=True).set_index("version").loc[
        [v for v in ["old", "medaka3d", "m3dfull"] if v in set(pd.concat(tables)["version"])]]
    display(T_SWITCH)
    display(error_table(STATE).query("kind == 'swap'"))

,switches,into wrong,back to right,only this (±1 s),with F,with M1,with M2,longest right run s,ambiguous %
version,,,,,,,,,
old,55,41,14,26,49,52,10,365.0,0.03
medaka3d,54,51,3,22,47,50,12,229.8,1.06
m3dfull,51,27,24,30,44,50,8,473.3,0.21


,kind,old entries,old fish-frames,old %,m3dfull entries,m3dfull fish-frames,m3dfull %,only old,only m3dfull,both
4,swap,36,656433,63.525,39,144434,13.977,548959,36960,107474


## 步骤 10 — λ 扫描（7 个 λ 的切换次数）

- 每个 λ 的 `C34-label` 胸鳍锚点直接和审阅结果逐帧匹配，在同一帧继承名字，按步骤 9 的同一规则数切换 —— 不用为每个 λ 生成审阅包
- λ = 0.767 这一行应和步骤 9 的 m3dfull 基本一致（这里用的是链的原始锚点，步骤 9 用的是审阅包重新三角化的胸鳍），可作自检
- medaka-3d 的规则：**结论只取所有 λ 都成立的部分**

In [10]:
if STATE is None:
    print("还没有对比数据（步骤 8）")
else:
    t0 = time.time()
    T_LAMBDA = M3C.lambda_switch_table(EXPERIMENT, STATE, m3d=M3D)
    display(T_LAMBDA)
    print(f"({time.time() - t0:.0f} s)")

,lambda_mm,switches,into wrong,with F,with M1,with M2,coverage %
0,0.000,107,101,84,102,36,99.99
1,0.100,63,59,52,61,13,99.99
2,0.250,59,45,50,56,12,99.99
3,0.500,55,42,47,53,10,99.99
4,0.767,51,27,44,50,8,99.98
5,2.000,49,38,43,49,6,99.98
6,8.000,49,30,39,49,10,99.97


(3 s)


## 步骤 11 — 几何（不依赖参考）

- 每个版本用自己的轨迹：胸鳍覆盖率、相邻帧胸鳍跳变 > 10 mm / > 30 mm 的次数（瞬移或没修好的换 ID）、节点来源（三角化 / 单视角 / 身体模型）
- `reviewed` 一行包含人工修正，只作背景，不参与比较
- 表 2：各类分歧（位置差 > 5 mm、头尾差 > 5 mm、只有一方有 3D）的条数和帧数 —— 偏向 old（见开头说明），要在 GUI 里对着视频判定

In [11]:
if STATE is None:
    print("还没有对比数据（步骤 8）")
else:
    display(geometry_table(STATE))
    display(summary(STATE))

,version,kind,coverage %,steps > 10 mm,steps > 30 mm,nodes triangulated %,nodes one view %,nodes body model %
0,reviewed,final,100.00,0,0,98.63,0.24,0.08
1,old,initial,100.00,15,0,99.54,0.35,0.11
2,m3dfull,initial,99.95,89,29,99.59,0.39,0.03


,other,group,kind,entries,frames,max_mm,A right,B right,both wrong,both OK
0,m3dfull,coverage,only m3dfull,2,200,0.000000,0,0,0,0
1,m3dfull,coverage,only reviewed,15,1770,0.000000,0,0,0,0
2,m3dfull,geometry,nodes,1434,19185,196.603274,0,0,0,0
3,m3dfull,geometry,position,313,3176,235.161713,0,0,0,0
4,m3dfull,identity,swap,48,150885,294.714502,0,0,0,0
5,old,geometry,nodes,1360,13955,67.454168,0,0,0,0
6,old,geometry,position,356,2159,36.330972,0,0,0,0
7,old,identity,swap,36,656433,297.748949,0,0,0,0


## 步骤 12 — 打开对比 GUI（“3D 重建对比”模式）

- 7/8/9 切换版本对；A/B/W/O 判定分歧（写在 `recon_reviewed_old_m3dfull/recon_verdicts.csv`，不碰审阅记录）

In [12]:
if STATE is None:
    print("还没有对比数据（步骤 8）")
else:
    print("在 PowerShell 里运行：")
    print(f'  cd "{REPO}"')
    print(f'  & "{sys.executable}" -m scripts.review.compare_gui "{REVIEW_A}" --mode recon --recon-dir "{STATE.recon_dir}"')
    if OPEN_GUI:
        subprocess.Popen([sys.executable, "-m", "scripts.review.compare_gui", str(REVIEW_A), "--mode", "recon",
                          "--recon-dir", str(STATE.recon_dir)], cwd=REPO)

在 PowerShell 里运行：
  cd "C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline"
  & "C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\.venv\Scripts\python.exe" -m scripts.review.compare_gui "\\bucket.oist.jp\bucket\StephensU\medaka\review_packages\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed" --mode recon --recon-dir "\\bucket.oist.jp\bucket\StephensU\medaka\review_packages\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed\recon_reviewed_old_m3dfull"


## 结论（2026-10-28，整条链 c27→c34 用时约 1 h 51 min：c27 28 分钟，c28 7 个 λ 共 74 分钟，c33/c30/c34 共 9 分钟）

**身份：两个完整流程差不多，medaka-3d 没有稳定地更好。**

| | old | medaka3d（桥接，只换逐帧配对） | m3dfull（λ = 0.767） |
|---|---|---|---|
| 切换次数 | 55 | 54 | 51 |
| 换错 / 换回来 | 41 / 14 | 51 / 3 | 27 / 24 |
| 只有自己换（±1 s） | 26 | 22 | 30 |
| 涉及 F | 49 | 47 | 44 |
| 最长连续正确 | 365 s | 230 s | 473 s |
| 名字错的鱼-帧 | 63.5 % | — | 14.0 % |

- **λ 扫描**：切换次数随 λ 变化为 107（λ = 0）、63、59、55、**51**（0.767）、49、49。只有 λ ≥ 0.767 时比 old 的 55 少；λ = 0.5 时持平，λ 更小时更差。按 medaka-3d 自己的规则（结论要对所有 λ 都成立），"m3dfull 切换更少"**不成立**。而且 51 比 55 只少 4 次，本身就很小。
- **拆解差异**：只换逐帧配对（桥接）几乎没变（54 次），而且换错之后很少换回来（3 次）。m3dfull 的改进主要来自后面几步（2D 先追踪、Viterbi、修补）：换错之后更常自己换回来（24 次 vs old 的 14 次）。所以名字错的鱼-帧从 63.5 % 降到了 14 %。但这个指标主要反映第一次永久换错发生得多早，不是主指标。
- **难点没变**：两个流程的切换几乎都涉及 F（44–49 次），主要发生在 F 和 M1 相遇的时候（with M1 约 50 次）。纯几何加时间连续性的方法在这里都会出错，需要外观识别（Line B）来区分。

**几何：old 更平滑，m3dfull 有更多瞬移。**
- 相邻帧胸鳍跳变超过 10 mm：old 15 次，m3dfull 89 次；超过 30 mm：old 0 次，m3dfull 29 次。
- 位置分歧的最大值：m3dfull 235 mm，old 36 mm。
- 覆盖率：m3dfull 99.95 %，old 100 %（old 插值补洞，medaka-3d 留空）。
- 这一部分偏向 old（审阅是在 old 的结果上修改的），而且大跳变可能是 m3dfull 槽之间的短暂互换。要在 GUI（步骤 12）里对着视频判定后才能下结论。

**限制**：
- 检测：两个流程都用 `--max_instances 6` 的同一份 `.slp`，而 medaka-3d 原本不限实例数。
- c30 v4 的阈值（2.5 mm / 30 mm）是在 K_1_2 上调出来的，没有为 P 组调整。
- 只有一个实验（P_1_1）、一份审阅。

**建议**：
- 现在没有理由把生产流程换成 medaka-3d。
- 如果想用 medaka-3d，可以借用它"换错之后换回来"的机制（时间上的全局优化），同时保留 old 的平滑几何。
- 身份问题的关键还是 F 和 M 相遇时的外观识别。
